# Notebook 05 - ML Baseline
**Proyek:** Pulsevera - Predict, Prevent, Prevail  
**Role:** AI Engineer  
**Tujuan:** Melatih 3 model Machine Learning baseline (Logistic Regression, Decision Tree, Random Forest) untuk memprediksi risiko penyakit jantung, sekaligus menyiapkan data untuk proses training.

---

## Setup & Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score,
    mean_absolute_error
)

import joblib

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print('Semua library berhasil diimport.')

Semua library berhasil diimport.


In [2]:
# Tentukan lokasi folder data dan output model
BASE_DIR   = Path('..')
DATA_DIR   = BASE_DIR / 'data' / 'final'
MODELS_DIR = BASE_DIR / 'ml-api' / 'models'

# Pastikan folder models sudah ada
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data   : {DATA_DIR}')
print(f'Models : {MODELS_DIR}')

Data   : ../data/final
Models : ../ml-api/models


---
## Step 1: Load Data
Membaca file CSV hasil preprocessing dari tim Data Science.

In [3]:
# -Load 4 file CSV dari folder data/final
# File  bersih dari Data Science
X_train = pd.read_csv(DATA_DIR / 'X_train.csv')
X_test  = pd.read_csv(DATA_DIR / 'X_test.csv')
y_train = pd.read_csv(DATA_DIR / 'y_train.csv').squeeze()
y_test  = pd.read_csv(DATA_DIR / 'y_test.csv').squeeze()

# Tampilkan shape masing-masing
print('=== SHAPE DATA ===')
print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_test  : {y_test.shape}')

=== SHAPE DATA ===
X_train : (356105, 46)
X_test  : (89027, 46)
y_train : (356105,)
y_test  : (89027,)


In [4]:
# Verifikasi tidak ada missing value
print('=== MISSING VALUES ===')
print(f'X_train : {X_train.isnull().sum().sum()}')
print(f'X_test  : {X_test.isnull().sum().sum()}')

# Cek distribusi label untuk konfirmasi class imbalance
print('\n=== DISTRIBUSI LABEL y_train ===')
print(y_train.value_counts())
print(f'\n% Positif (berisiko): {y_train.mean()*100:.2f}%')

=== MISSING VALUES ===
X_train : 0
X_test  : 0

=== DISTRIBUSI LABEL y_train ===
HadHeartAttack
0    336019
1     20086
Name: count, dtype: int64

% Positif (berisiko): 5.64%


---
## Step 2: Perbaiki Tipe Data
5 kolom bertipe `bool` harus dikonversi ke `float32` agar tidak error saat diproses scikit-learn maupun TensorFlow.

Kolom ini adalah hasil one-hot encoding dari kolom `RaceEthnicityCategory` yang dilakukan oleh tim Data Science.

In [5]:
# Deteksi dan konversi kolom bool ke float32
# Kolom Race_* bertipe bool karena hasil one-hot encoding dari Data Science
bool_cols = X_train.select_dtypes(include='bool').columns.tolist()
print(f'Kolom bool yang ditemukan ({len(bool_cols)} kolom):')
print(bool_cols)

X_train[bool_cols] = X_train[bool_cols].astype('float32')
X_test[bool_cols]  = X_test[bool_cols].astype('float32')

print('\nKonversi selesai. Tipe data sekarang:')
print(X_train.dtypes.value_counts())

Kolom bool yang ditemukan (5 kolom):
['Race_Black only, Non-Hispanic', 'Race_Hispanic', 'Race_Multiracial, Non-Hispanic', 'Race_Other race only, Non-Hispanic', 'Race_White only, Non-Hispanic']

Konversi selesai. Tipe data sekarang:
int64      32
float64     9
float32     5
Name: count, dtype: int64


---
## Step 3: Normalisasi Data (StandardScaler)
Menyamakan skala semua fitur agar fitur dengan angka besar (BMI, berat badan) tidak mendominasi model.

> **Penting:** Scaler hanya di-`fit` pada X_train. X_test cukup di-`transform` menggunakan parameter yang sama — tidak boleh di-fit ulang.

In [6]:
# Inisialisasi dan fit StandardScaler pada X_train
# fit_transform: scaler belajar mean & std dari data training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Scaling selesai.')
print(f'Mean X_train_scaled (harusnya mendekati 0): {X_train_scaled.mean():.4f}')
print(f'Std  X_train_scaled (harusnya mendekati 1): {X_train_scaled.std():.4f}')

Scaling selesai.
Mean X_train_scaled (harusnya mendekati 0): 0.0000
Std  X_train_scaled (harusnya mendekati 1): 1.0000


In [7]:
# Simpan scaler ke file
# Scaler WAJIB disimpan karena akan dipakai lagi saat inference di FastAPI
scaler_path = MODELS_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler berhasil disimpan ke: {scaler_path}')

Scaler berhasil disimpan ke: ../ml-api/models/scaler.pkl


---
## Step 4: Tangani Class Imbalance (SMOTE)
Data training sangat tidak seimbang (~5.64% positif). SMOTE membuat data sintetis untuk kelas minoritas agar model tidak bias ke kelas negatif.

> **Penting:** SMOTE hanya diterapkan pada X_train. X_test tidak boleh disentuh agar evaluasi tetap mencerminkan kondisi nyata.

In [8]:
# Terapkan SMOTE hanya pada data training untuk mengatasi class imbalance
# sampling_strategy=0.3 artinya kelas positif akan menjadi 30% dari kelas negatif
smote = SMOTE(random_state=42, sampling_strategy=0.3)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print('=== DISTRIBUSI SETELAH SMOTE ===')
print(pd.Series(y_train_res).value_counts())
print(f'\nShape X_train setelah SMOTE: {X_train_res.shape}')

=== DISTRIBUSI SETELAH SMOTE ===
HadHeartAttack
0    336019
1    100805
Name: count, dtype: int64

Shape X_train setelah SMOTE: (436824, 46)


---
## Step 5: Training 3 Model ML Baseline
Melatih Logistic Regression, Decision Tree, dan Random Forest dengan data yang sudah disiapkan.

In [9]:
# Definisikan 3 model baseline
# Tidak pakai class_weight='balanced' karena class imbalance sudah ditangani SMOTE di atas
# random_state=42 agar hasil bisa direproduksi kapan saja
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10,       # batasi kedalaman agar tidak overfitting
        min_samples_leaf=50,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        random_state=42,
        n_jobs=-1  # pakai semua core CPU agar lebih cepat
    ),
}

print(f'Total model yang akan dilatih: {len(models)}')

Total model yang akan dilatih: 3


In [10]:
# Training semua model, lalu evaluasi dengan X_test asli
results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_res, y_train_res)

    y_pred  = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    results[name] = {
        'model'   : model,
        'y_pred'  : y_pred,
        'y_proba' : y_proba,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc' : roc_auc_score(y_test, y_proba),
        'mae'     : mean_absolute_error(y_test, y_proba),
        'report'  : classification_report(y_test, y_pred, output_dict=True)
    }

print('Training semua model selesai.')

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training semua model selesai.


---
## Step 6: Evaluasi & Perbandingan Model
Menampilkan hasil semua model dalam satu tabel untuk memudahkan perbandingan dan pemilihan model terbaik.

In [11]:
# Buat tabel perbandingan semua model
# Recall kelas 1 (positif) adalah metrik prioritas utama
comparison = []
for name, res in results.items():
    report = res['report']
    comparison.append({
        'Model'              : name,
        'Accuracy'           : round(res['accuracy'], 4),
        'Recall (kelas 1)'   : round(report['1']['recall'], 4),
        'Precision (kelas 1)': round(report['1']['precision'], 4),
        'F1 (kelas 1)'       : round(report['1']['f1-score'], 4),
        'ROC-AUC'            : round(res['roc_auc'], 4),
        'MAE'                : round(res['mae'], 4),
    })

df_comparison = pd.DataFrame(comparison)
print('=== PERBANDINGAN MODEL ===')
df_comparison

=== PERBANDINGAN MODEL ===


,Model,Accuracy,Recall (kelas 1),Precision (kelas 1),F1 (kelas 1),ROC-AUC,MAE
0,Logistic Regression,0.9259,0.5430,0.3880,0.4526,0.8821,0.1565
1,Decision Tree,0.9429,0.3624,0.4915,0.4172,0.8652,0.1024
2,Random Forest,0.9457,0.2559,0.5399,0.3472,0.8712,0.0936


In [12]:
# Tampilkan classification report lengkap per model
for name, res in results.items():
    print(f'=== {name} ===')
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Tidak Berisiko (0)', 'Berisiko (1)']
    ))
    print(f'ROC-AUC : {res["roc_auc"]:.4f}')
    print(f'MAE     : {res["mae"]:.4f}')
    print()

=== Logistic Regression ===
                    precision    recall  f1-score   support

Tidak Berisiko (0)       0.97      0.95      0.96     84005
      Berisiko (1)       0.39      0.54      0.45      5022

          accuracy                           0.93     89027
         macro avg       0.68      0.75      0.71     89027
      weighted avg       0.94      0.93      0.93     89027

ROC-AUC : 0.8821
MAE     : 0.1565

=== Decision Tree ===
                    precision    recall  f1-score   support

Tidak Berisiko (0)       0.96      0.98      0.97     84005
      Berisiko (1)       0.49      0.36      0.42      5022

          accuracy                           0.94     89027
         macro avg       0.73      0.67      0.69     89027
      weighted avg       0.94      0.94      0.94     89027

ROC-AUC : 0.8652
MAE     : 0.1024

=== Random Forest ===
                    precision    recall  f1-score   support

Tidak Berisiko (0)       0.96      0.99      0.97     84005
      Beris

---
## Step 6b: Threshold Tuning
Recall model terbaik (Logistic Regression) masih 54% — belum memenuhi target ≥ 70%.

Kita turunkan threshold prediksi dari default 0.5 untuk meningkatkan Recall.
> **Trade-off:** Recall naik → Precision turun (lebih banyak false alarm). Ini wajar untuk konteks medis.

In [13]:
from sklearn.metrics import recall_score, precision_score, f1_score

# Coba berbagai threshold untuk Logistic Regression (model terbaik)
y_proba_lr = results['Logistic Regression']['y_proba']
thresholds  = [0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]

tuning_rows = []
for thresh in thresholds:
    y_pred_thresh = (y_proba_lr >= thresh).astype(int)
    tuning_rows.append({
        'Threshold' : thresh,
        'Accuracy'  : round(accuracy_score(y_test, y_pred_thresh), 4),
        'Recall'    : round(recall_score(y_test, y_pred_thresh), 4),
        'Precision' : round(precision_score(y_test, y_pred_thresh), 4),
        'F1'        : round(f1_score(y_test, y_pred_thresh), 4),
    })

df_tuning = pd.DataFrame(tuning_rows)
print('=== THRESHOLD TUNING — Logistic Regression ===')
print(df_tuning.to_string(index=False))
print('\nTarget: Recall >= 0.70 dengan Precision >= 0.20')

=== THRESHOLD TUNING — Logistic Regression ===
 Threshold  Accuracy  Recall  Precision     F1
      0.50    0.9259  0.5430     0.3880 0.4526
      0.40    0.9074  0.6157     0.3289 0.4287
      0.30    0.8714  0.7019     0.2615 0.3810
      0.25    0.8388  0.7495     0.2233 0.3441
      0.20    0.7919  0.8023     0.1868 0.3031
      0.15    0.7206  0.8664     0.1524 0.2592
      0.10    0.6062  0.9275     0.1184 0.2100

Target: Recall >= 0.70 dengan Precision >= 0.20


In [14]:
# Pilih threshold terbaik: Recall >= 0.70 dengan Precision >= 0.20
best_thresh = None
for row in tuning_rows:
    if row['Recall'] >= 0.70 and row['Precision'] >= 0.20:
        best_thresh = row['Threshold']
        break

if best_thresh is None:
    # Kalau tidak ada yang memenuhi keduanya, pilih yang Recall-nya paling mendekati 0.70
    best_thresh = min(tuning_rows, key=lambda r: abs(r['Recall'] - 0.70))['Threshold']
    print(f'Tidak ada threshold yang memenuhi semua kriteria.')
    print(f'Threshold terpilih (mendekati Recall 0.70): {best_thresh}')
else:
    print(f'Threshold terpilih: {best_thresh}')

# Simpan threshold ke file agar dipakai saat inference di FastAPI
joblib.dump(best_thresh, MODELS_DIR / 'threshold.pkl')
print(f'Threshold disimpan ke: {MODELS_DIR / "threshold.pkl"}')

Threshold terpilih: 0.3
Threshold disimpan ke: ../ml-api/models/threshold.pkl


---
## Step 7: Pilih Model Terbaik & Simpan
Model terbaik dipilih berdasarkan Recall tertinggi pada kelas positif (kelas 1), lalu disimpan ke file untuk dipakai di FastAPI.

In [15]:
# Pilih model terbaik berdasarkan Recall kelas 1 tertinggi
# tapi dengan syarat: Precision kelas 1 minimal 20%
# Tanpa syarat ini, model yang selalu prediksi "berisiko" bisa menang dengan Recall=1.0

PRECISION_THRESHOLD = 0.20

candidates = {
    name: res for name, res in results.items()
    if res['report']['1']['precision'] >= PRECISION_THRESHOLD
}

if not candidates:
    print('Tidak ada model yang memenuhi threshold precision. Turunkan PRECISION_THRESHOLD.')
else:
    best_name  = max(candidates, key=lambda name: candidates[name]['report']['1']['recall'])
    best_model = results[best_name]['model']

    print(f'Model terbaik    : {best_name}')
    print(f'Recall (kelas 1) : {results[best_name]["report"]["1"]["recall"]:.4f}')
    print(f'Precision (kelas 1): {results[best_name]["report"]["1"]["precision"]:.4f}')
    print(f'ROC-AUC          : {results[best_name]["roc_auc"]:.4f}')
    print(f'MAE              : {results[best_name]["mae"]:.4f}')

Model terbaik    : Logistic Regression
Recall (kelas 1) : 0.5430
Precision (kelas 1): 0.3880
ROC-AUC          : 0.8821
MAE              : 0.1565


In [16]:
# Simpan model terbaik ke file .pkl
model_path = MODELS_DIR / 'pulsevera_ml_model.pkl'
joblib.dump(best_model, model_path)
print(f'Model disimpan ke: {model_path}')

# Simpan urutan kolom fitur
# Urutan kolom WAJIB sama persis antara training dan inference
# Kalau berbeda, prediksi akan salah meski modelnya benar
feature_order = X_train.columns.tolist()
joblib.dump(feature_order, MODELS_DIR / 'feature_order.pkl')
print(f'Feature order disimpan ke: {MODELS_DIR / "feature_order.pkl"}')

Model disimpan ke: ../ml-api/models/pulsevera_ml_model.pkl
Feature order disimpan ke: ../ml-api/models/feature_order.pkl


---
## Ringkasan Notebook 05
Catat hasil akhir dan file yang dihasilkan dari notebook ini.

In [17]:
# Ringkasan hasil
print('=== RINGKASAN ===')
print(f'Dataset         : {X_train.shape[0]:,} training | {X_test.shape[0]:,} test')
print(f'Jumlah fitur    : {X_train.shape[1]} kolom')
print(f'Setelah SMOTE   : {X_train_res.shape[0]:,} baris training')
print(f'Model terbaik   : {best_name}')
print()
print('File yang dihasilkan:')
print('  - ml-api/models/scaler.pkl')
print('  - ml-api/models/pulsevera_ml_model.pkl')
print('  - ml-api/models/feature_order.pkl')

=== RINGKASAN ===
Dataset         : 356,105 training | 89,027 test
Jumlah fitur    : 46 kolom
Setelah SMOTE   : 436,824 baris training
Model terbaik   : Logistic Regression

File yang dihasilkan:
  - ml-api/models/scaler.pkl
  - ml-api/models/pulsevera_ml_model.pkl
  - ml-api/models/feature_order.pkl
